In [ ]:
!pip install phonenumbers
!pip install supabase python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.0 MB/s eta 0:00:00


In [ ]:
import gspread
import pandas as pd
import numpy as np
from google.colab import drive
import re
import phonenumbers
from supabase import create_client, Client
import dotenv
from google.colab import userdata
from dotenv import load_dotenv
import json
import os


ModuleNotFoundError: No module named 'phonenumbers'

In [ ]:
def extract_google_sheet ():
  drive.mount("/content/drive", force_remount=True)
  gc =gspread.service_account('/@@@@@@@@@@@@@@.json')
  sheet = gc.open_by_url("https://docs.google.com//spreadsheets/d/19hqhMksx5LZqP3QR0i60W7-OMIXKBkQvFC4lBVIdDyg/").sheet1
  data = sheet.get_all_values()
  df = pd.DataFrame(data[1:], columns=data[0])
  return df,sheet


In [ ]:
def clean_phone(x):
    try:
        num = phonenumbers.parse(str(x), "FR")
        if phonenumbers.is_valid_number(num):
            return phonenumbers.format_number(num, phonenumbers.PhoneNumberFormat.INTERNATIONAL)
    except phonenumbers.NumberParseException:
        return None
    return None

In [ ]:
def check_superficie(val):
    if pd.isna(val):
        return 'non'
    s = str(val)
    nombres = re.findall(r'\d+', s)
    for n in nombres:
        if int(n) >= 20:
            return 'oui'
    return 'non'

In [ ]:
def clean_montant(x):
    if pd.isna(x):
        return None
    x = str(x).replace('\xa0', ' ')
    x = x.replace('€', '').replace(' ', '').replace('\xa0', '')
    x = x.replace(',', '.')
    pattern = r'\d+(?:[ ]\d{3})*(?:[.,]\d+)?'
    matches = re.findall(pattern, x)

    if not matches:
        return None

    total = 0.0
    for m in matches:
        num = m.replace(' ', '').replace(',', '.')
        total += float(num)

    return total

In [ ]:
def get_periode(x, montant, seuil=800):
    if pd.isna(x):
        return None
    x = str(x).upper()

    if re.search(r'\bMOIS\b', x):
        return 'mois'
    if re.search(r'\bANS?\b|ANNU|ANNÉE|ANNEE', x):
        return 'an'

    if pd.isna(montant):
        return None
    return 'an' if montant > seuil else 'mois'


In [ ]:
def clean_mode_chauffage(x):

    if pd.isna(x):
        return "Inconnu"

    x = str(x).lower().strip()


    if x in ["", "---", "empty", "nan", "none"]:
        return "Inconnu"

    x = re.sub(r"\d+", "", x)
    x = x.replace("+", " ")
    x = x.replace("/", " ")
    x = x.replace("-", " ")

    if "elec" in x or "elect" in x:
        if "pellet" in x or "pelet" in x or "bois" in x or "clim" in x:
            return "Mixte"
        return "Electricite"

    if "gaz" in x:
        return "Gaz"

    if "fioul" in x or "mazoute" in x:
        return "Fioul"

    if "pac" in x or "pompe" in x:
        return "Pompe_a_chaleur"

    if "pellet" in x or "pelet" in x:
        return "Pellet"

    if "bois" in x:
        if "clim" in x:
            return "Mixte"
        return "Bois"

    if "clim" in x:
        return "Climatisation"

    return "Autre"

In [ ]:
def clean_data(df):

    df['telephone'] = df['telephone'].apply(clean_phone)
    df['Mobile'] = df['Mobile'].apply(clean_phone)
    df['Revenu'] = df['Revenu'].str.replace(r'[^\d.-]', '', regex=True)
    df['Crédits en cours'] = df['Crédits en cours'].str.replace(r'[^\d.-]', '', regex=True)
    df['Heure RDV'] = df['Heure RDV'].replace('30/12/1899', '-')
    df['Superficie Habitable'] =  df['Superficie Habitable'].str.extract(r'(\d+)')
    df['Année de construction'] = df['Année de construction'].str.extract(r'(\d+)')
    df = df.drop_duplicates(subset=["telephone", "Mobile"], keep="first").copy()
    df['Heure RDV'] = df['Heure RDV'].replace('30/12/1899', '-')
    df['Année de construction'] = df['Année de construction'].str.extract(r'(\d+)')
    h = (
        df['Heure RDV']
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace('H', ':', regex=False)
        .str.rstrip(':')
        .replace({'-': None, '': None, 'NAN': None})
    )
    h = h.str.replace(r'^(\d{1,2})$', r'\1:00:00', regex=True)
    h = h.str.replace(r'^(\d{1,2}):(\d{1,2})$', r'\1:\2:00', regex=True)

    df['Heure RDV'] = pd.to_datetime(h, format='%H:%M:%S', errors='coerce').dt.strftime('%H:%M:%S')
    df['Superficie_valide'] = df['Superficie Toiture (20 m²)'].apply(check_superficie)
    df['Date de rdv'] = pd.to_datetime(df['Date de rdv'], format='%d/%m/%Y', errors='coerce')

    cols = ['NOM','prenom','ville ','Statut marital', 'Secteur d’Activité (Monsieur)','AGENT',
        'Situation Professionnelle (Mr)','Secteur d’Activité (Mme)','Situation Professionnelle (Mme)','Situation Professionnelle (Mme)',
        'Type Contrat (Mr)','Type Contrat (Mme)','Mode de Chauffage']

    for col in cols:
        if col not in df.columns:
            print(f" Colonne absente : {col}")
            continue
        df[col] = df[col].str.strip().str.lower()
    pd.set_option('display.max_columns', None)
    df['AGENT'] = df['AGENT'].str.strip().str.upper()
    df['Confirmateur'] = df['Confirmateur'].str.strip().str.upper()
    df['montant'] = df['Facture Chauffage'].apply(clean_montant)
    df['periode'] = df.apply(
    lambda row: get_periode(row['Facture Chauffage'], row['montant']),
    axis=1
    )
    df['facture_annuelle'] = np.where(df['periode'] == 'mois',df['montant'] * 12,df['montant'])
    df=df.rename(columns=str.lower)
    df.columns = df.columns.str.strip()

    df = df.loc[:, df.columns != '']

    rename_map = {
          'status': 'status',
          'confirmateur': 'confirmateur',
          'injecté': 'injecte',
          "motif d'annulation": 'motif_annulation',
          'appel': 'date_appel',
          'nom': 'nom',
          'prenom': 'prenom',
          'adresse': 'adresse',
          'code postal': 'code_postal',
          'ville': 'ville',
          'telephone': 'telephone',
          'mobile': 'mobile',
          'adresse mail': 'adresse_mail',
          'statut marital': 'statut_marital',
          'enfants à charge': 'enfants_a_charge',
          'crédits en cours': 'credits_en_cours',
          'revenu': 'revenu',
          'situation professionnelle (mr)': 'situation_professionnelle_mr',
          'type contrat (mr)': 'type_contrat_mr',
          'secteur d’activité (monsieur)': 'secteur_activite_mr',
          'situation professionnelle (mme)': 'situation_professionnelle_mme',
          'secteur d’activité (mme)': 'secteur_activite_mme',
          'type contrat (mme)': 'type_contrat_mme',
          'superficie toiture (20 m²)': 'superficie_toiture',
          'zone classée': 'zone_classee',
          'maison plain-pied': 'maison_plain_pied',
          'mode de chauffage': 'mode_chauffage',
          'facture chauffage': 'facture_chauffage',
          'propriétaire': 'proprietaire',
          'age (mr)': 'age_mr',
          'age (mme)': 'age_mme',
          'superficie habitable': 'superficie_habitable',
          'année de construction': 'annee_construction',
          'date de rdv': 'date_rdv',
          'heure rdv': 'heure_rdv',
          'type de la fiche ( rdv ou ficheè )': 'type_fiche',
          'AGENT': 'agent',
          'commentaire': 'commentaire',
          'montant': 'montant',
          'facture_annuelle': 'facture_annuelle',
          'superficie_valide': 'superficie_valide',
      }

    df = df.rename(columns=rename_map)
    df["mode_chauffage"] = df["mode_chauffage"].apply(clean_mode_chauffage)
    df['revenu'] = pd.to_numeric(df['revenu'], errors='coerce')
    for col in ['enfants_a_charge', 'age_mr', 'age_mme']:
      df[col] = pd.to_numeric(df[col], errors='coerce')
      df['date_rdv'] = pd.to_datetime(df['date_rdv'], errors='coerce').dt.strftime('%Y-%m-%d')
      df['heure_rdv'] = pd.to_datetime(df['heure_rdv'], errors='coerce').dt.strftime('%H:%M:%S')
      df['date_appel'] = pd.to_datetime(df['date_appel'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
      df= df.replace({np.nan: None, pd.NaT: None})
      df = df.where(pd.notnull(df), None)
    return df


In [ ]:
df,_ = extract_google_sheet()
df = clean_data(df)
df.tail()

Mounted at /content/drive


/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['heure_rdv'] = pd.to_datetime(df['heure_rdv'], errors='coerce').dt.strftime('%H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:103: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date_appel'] = pd.to_datetime(df['date_appel'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['heure_rdv'] = pd.to_datetime(df['heure_rdv'], errors='coerce').dt.strftime('%H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Cou

,status,confirmateur,injecte,motif_annulation,date_appel,nom,prenom,adresse,code_postal,ville,telephone,mobile,adresse_mail,statut_marital,enfants_a_charge,credits_en_cours,revenu,situation_professionnelle_mr,type_contrat_mr,secteur_activite_mr,situation_professionnelle_mme,secteur_activite_mme,type_contrat_mme,superficie_toiture,zone_classee,maison_plain_pied,mode_chauffage,facture_chauffage,proprietaire,age_mr,age_mme,superficie_habitable,annee_construction,date_rdv,heure_rdv,type_fiche,agent,commentaire,superficie_valide,montant,periode,facture_annuelle
5428,Conf brut,AUBRY,,,None,graveline,evelyse,30 Rue du Moulin,59148,flines lez raches,+33 6 27 67 01 80,+33 6 27 67 01 80,evemanudu59148@live.fr,mariée,3.0,1400,2850.0,actif,couvreur,/,active,auxilaire,/,20m²ok,,,Pompe_a_chaleur,300 euro,Oui,None,None,140,None,2026-09-08,18:30:00,Fiche,HASSIB MUSTAPHA,"Rdv pv confirmé par Mme, chauffage pac, factur...",oui,300.0,mois,3600.0
5429,RDV de principe,AICHA,,,2026-09-03 16:48:39,joud bouvad,stephane,73 Chem. Simandre,38260,pommier de beaurepaire,+33 6 07 26 23 83,+33 6 07 26 23 83,,couple,0.0,800,1800.0,actif,rénovation,a son compte,sans activité,,,20 m²,Non,Oui,Pompe_a_chaleur,+1000€,Oui,58.0,54.0,240,None,2026-09-15,18:30:00,RDV,ABDERRAHMAN OMARY,Rdv PV pris avec Mr (Equipé 3kwc depuis +10 an...,oui,1000.0,an,1000.0
5430,,MARIUS,,,2026-09-03 16:59:36,vincent,robert,theillet,71520,saint-pierre-le-vieux,+33 6 11 14 02 02,+33 6 11 14 02 02,,seul,0.0,0,1000.0,retraite-actif,immobilier,retraite-actif,---,---,---,20,Non,Oui,Bois,1000,Oui,72.0,None,220,2,2026-09-07,10:00:00,RDV,KHAYATE KHADIJA,RDV PV pour résidence secondaire qui deviendra...,oui,1000.0,an,1000.0
5431,RAPPEL,YASSINE,,,2026-09-03 17:03:35,bossy,christelle,5 Rue du Bois d'Alou,28360,meslay-le-vidame,+33 6 20 07 73 73,+33 6 20 07 73 73,-,couple,0.0,900,1900.0,actif,salarié,cdi,active,salariée,cdi,20,Non,Non,Autre,Elect 100€/mois,Oui,-66.0,-66.0,None,None,2026-09-08,18:30:00,RDV,ABDERRAHMAN OMARY,Veut voir avec Mr si d'accord rap demain 04/09...,oui,100.0,mois,1200.0
5432,Conf brut,AUBRY,,,2026-09-03 17:18:18,lacosse,mathilde,"133 Rue de la Conche,",38390,"porcieu-amblagnieu, france",+33 6 98 12 00 33,+33 6 98 12 00 33,.,couple,2.0,900,2200.0,actif,scierie,cdi,active,fonctionnaire,cdi,.,Non,Non,Bois,220,Oui,-70.0,-70.0,None,None,2026-09-08,18:30:00,RDV,KADDI AYOUB,"Rdv pv confirmé par Mme, chauffage bois, factu...",non,220.0,mois,2640.0


In [ ]:
from google.colab import userdata
from supabase import create_client, Client
url = userdata.get('SUPABASE_URL')
key = userdata.get('SUPABASE_KEY')
supabase: Client = create_client(url, key)

In [ ]:
def prepare_records(df):
  records = df.to_dict(orient='records')
  return records


In [ ]:
def insert_to_supabase(records):
    if not records:
        print("No records to insert.")
        return

    try:
        response = supabase.table("rendez_vous").insert(records).execute()
        print(f"Succès ! {len(response.data)} lignes insérées.")
    except Exception as e:
        print(f"Erreur lors de l'insertion : {e}")

In [ ]:
records= prepare_records(df)
insert_to_supabase(records)

Succès ! 5056 lignes insérées.


In [ ]:
STATE_FILE = "sync_state.json"

def load_last_synced_row():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE, "r") as f:
            return json.load(f).get("last_synced_row", 0)
    return 0

def save_last_synced_row(value):
    with open(STATE_FILE, "w") as f:
        json.dump({"last_synced_row": value}, f)

def sync_once():
    last_synced_row = load_last_synced_row()

    df, sheet = extract_google_sheet()
    rows = sheet.get_all_values()

    headers = rows[0]
    data = rows[1:]

    # remove fully empty rows (Sheets sometimes pads the grid)
    data = [row for row in data if any(cell.strip() for cell in row)]

    if len(data) > last_synced_row:
        new_rows = data[last_synced_row:]

        df = pd.DataFrame(new_rows, columns=headers)
        df = clean_data(df)
        records = prepare_records(df)
        insert_to_supabase(records)

        last_synced_row = len(data)
        save_last_synced_row(last_synced_row)

        print(f"{len(records)} new rows synchronized.")
    else:
        print("No new rows.")

In [ ]:
sync_once()

Mounted at /content/drive


/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['heure_rdv'] = pd.to_datetime(df['heure_rdv'], errors='coerce').dt.strftime('%H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:103: UserWarning: Parsing dates in %d/%m/%Y %H:%M:%S format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['date_appel'] = pd.to_datetime(df['date_appel'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['heure_rdv'] = pd.to_datetime(df['heure_rdv'], errors='coerce').dt.strftime('%H:%M:%S')
/tmp/ipykernel_1139/1478354679.py:102: UserWarning: Cou

NameError: name 'insert_to_supabase' is not defined

In [ ]:
print(df.shape)
print(df.index.min())
print(df.index.max())
print(len(df))

(5056, 42)
0
5416
5056
